In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Dataset Description:


* 'id'                  : id 
* 'Gender'              : Gender
* 'Age'                 : Age 
* 'Height'              : Height is in meter
* 'Weight'              : Weight is between 39 to 165
* 'family_history_with_overweight' : family history with overweight yes or no
* 'FAVC'                : Frequent consumption of high calorie food yes or no 
* 'FCVC'                : Frequency of consumption of vegetables yes or no 
* 'NCP'                 : Number of main meals
* 'CAEC'                : Consumption of food between meals
* 'SMOKE'               : yes or no 
* 'CH2O'                : Consumption of water daily
* 'SCC'                 : Calories consumption monitoring yes or no 
* 'FAF'                 : Physical activity frequency
* 'TUE'                 : Time using technology devices "How long using technology devices to track your health"
* 'CALC'                : Consumption of alcohol
* 'MTRANS'              : Transportation used
* 'NObeyesdad'          : Target Obesity 

# NObesity values:

* Underweight Less than 18.5
* Normal 18.5 to 24.9
* Overweight 25.0 to 29.9
* Obesity I 30.0 to 34.9
* Obesity II 35.0 to 39.9
* Obesity III Higher than 40

In [ ]:
# Importing required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
import xgboost as xgb
from lightgbm import LGBMClassifier
import lightgbm as lgb
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import roc_auc_score, roc_curve, auc
from sklearn.metrics import classification_report, confusion_matrix,accuracy_score

import warnings
warnings.filterwarnings('ignore')
# Disable LightGBM warnings
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)
import logging
logging.getLogger('lightgbm').setLevel(logging.INFO)
logging.getLogger('lightgbm').setLevel(logging.ERROR)

In [ ]:
# Loading data:
train_data = pd.read_csv(r"/kaggle/input/playground-series-s4e2/train.csv")
test_data = pd.read_csv(r"/kaggle/input/playground-series-s4e2/test.csv")
original_data = pd.read_csv(r"/kaggle/input/obesity-data/ObesityDataSet.csv")
sample_submission_data = pd.read_csv(r"/kaggle/input/playground-series-s4e2/sample_submission.csv")

# Shape of the data:
print("train_data :", train_data.shape)
print("test_data :", test_data.shape)
print("original_data :", original_data.shape)
print("sample_submission_data :", sample_submission_data.shape)

In [ ]:
train_data = train_data.drop("id", axis=1)
train_data = pd.concat([train_data, original_data], ignore_index=True)
train_data = train_data.drop_duplicates()
print("shape of the data :",train_data.shape)

In [ ]:
print(train_data['NObeyesdad'].value_counts())
sns.countplot(x='NObeyesdad',hue='Gender', data=train_data)
plt.xticks(rotation=60)
plt.show()

In [ ]:
num_cols = list(train_data.select_dtypes(exclude=['object']).columns)
cat_cols = list(train_data.select_dtypes(include=['object']).columns)

num_cols_test = list(test_data.select_dtypes(exclude=['object']).columns)
cat_cols_test = list(test_data.select_dtypes(include=['object']).columns)

num_cols_test = [col for col in num_cols_test if col not in ['id']]

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
train_data[num_cols] = scaler.fit_transform(train_data[num_cols])
test_data[num_cols_test] = scaler.transform(test_data[num_cols_test])

from sklearn.preprocessing import LabelEncoder
labelencoder = LabelEncoder()
train_data['NObeyesdad']=labelencoder.fit_transform(train_data['NObeyesdad'])

In [ ]:
#  object datatype columns encoding:
from sklearn.preprocessing import LabelEncoder
labelencoder = LabelEncoder()
object_columns = train_data.select_dtypes(include='object').columns.difference(['NObeyesdad'])

for col_name in object_columns:
    if train_data[col_name].dtypes=='object':
        train_data[col_name]=labelencoder.fit_transform(train_data[col_name]).astype(int)
        
for col_name in test_data.columns:
    if test_data[col_name].dtypes=='object':
        test_data[col_name]=labelencoder.fit_transform(test_data[col_name]).astype(int)

# Create heatmap
plt.figure(figsize=(15, 8))
sns.heatmap(train_data.corr(), annot=True, cmap='coolwarm')
plt.title('Correlation Heatmap')
plt.show()

In [ ]:
X = train_data.drop(['NObeyesdad'], axis=1)
y = train_data['NObeyesdad']
y = labelencoder.fit_transform(y)
X_test = test_data.drop(['id'],axis=1)
print(X.shape)
print(y.shape)
print(X_test.shape)
X_train, X_val, y_train, y_val = train_test_split(X,y,test_size=0.2,random_state=42)
X_train.shape , y_train.shape, X_val.shape, y_val.shape 

import optuna
# Define the objective function
def objective(trial):
    # Define hyperparameters to be optimized
    params = {
        "objective": "multiclass",
        "metric": "multi_logloss",
        "verbosity": -1,
        "boosting_type": "gbdt",
        "random_state": 42,
        "num_class": 7,
        'learning_rate': trial.suggest_loguniform('learning_rate', 0.01, 0.5),
        'n_estimators': trial.suggest_int('n_estimators', 100, 500),
        'lambda_l1': trial.suggest_loguniform('lambda_l1', 0.001, 0.1),
        'lambda_l2': trial.suggest_loguniform('lambda_l2', 0.001, 0.1),
        'max_depth': trial.suggest_int('max_depth', 5, 15),
        'colsample_bytree': trial.suggest_uniform('colsample_bytree', 0.5, 0.9),
        'subsample': trial.suggest_uniform('subsample', 0.5, 0.9),
        'min_child_samples': trial.suggest_int('min_child_samples', 10, 30)
    }
    
    # Split the data
    X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)
    
    # Train the LightGBM model
    model = lgb.LGBMClassifier(**params)
    model.fit(X_train, y_train, eval_set=[(X_val, y_val)])
    
    # Calculate accuracy
    y_pred = model.predict(X_val)
    accuracy = accuracy_score(y_val, y_pred)
    return accuracy

# Define the study
study = optuna.create_study(direction='maximize')

# Run the optimization
study.optimize(objective, n_trials=100)

# Print the best parameters and best value
print("Best parameters:", study.best_params)
print("Best accuracy:", study.best_value)

import lightgbm as lgb
from sklearn.model_selection import GridSearchCV

# Define the parameter grid
param_grid = {
    "learning_rate": [0.01, 0.1, 0.5],
    "n_estimators": [100, 200, 500],
    "lambda_l1": [0.001, 0.01, 0.1],
    "lambda_l2": [0.001, 0.01, 0.1],
    "max_depth": [5, 10, 15],
    "colsample_bytree": [0.5, 0.7, 0.9],
    "subsample": [0.5, 0.7, 0.9],
    "min_child_samples": [10, 20, 30]
}

# Initialize the LightGBM classifier
lgb_clf = lgb.LGBMClassifier(objective="multiclass",
                              metric="multi_logloss",
                              verbosity=-1,
                              boosting_type="gbdt",
                              random_state=42,
                              num_class=7)

# Perform GridSearchCV
grid_search = GridSearchCV(estimator=lgb_clf, param_grid=param_grid, cv=5, scoring='neg_log_loss', verbose=1)
grid_search.fit(X_train, y_train)

# Print the best parameters
print("Best parameters:", grid_search.best_params_)

In [ ]:
param1 = {'objective': 'multiclass',          
    'metric': 'multi_logloss',          
    'verbosity': -1,                    
    'boosting_type': 'gbdt',            
    'random_state': 42,       
    'num_class': 7,
    'learning_rate': 0.0327621905857854208, 
    'n_estimators': 500, 
    'lambda_l1': 0.009879324515507773, 
    'lambda_l2': 0.045092765238180027, 
    'max_depth': 10, 
    'colsample_bytree': 0.451686663982718, 
    'subsample': 0.9636469087931024, 
    'min_child_samples': 28}
#Best accuracy: 0.9166119500984898

In [ ]:
param = {'objective': 'multiclass',          
    'metric': 'multi_logloss',          
    'verbosity': -1,                    
    'boosting_type': 'gbdt',            
    'random_state': 42,       
    'num_class': 7,                     
    'learning_rate': 0.030962211546832760,  
    'n_estimators': 500,                
    'lambda_l1': 0.009667446568254372,  
    'lambda_l2': 0.04018641437301800,   
    'max_depth': 10,                    
    'colsample_bytree': 0.40977129346872643,  
    'subsample': 0.9535797422450176,   
    'min_child_samples': 26}

In [ ]:
model_lgb = lgb.LGBMClassifier(**param1)
model_lgb.fit(X_train, y_train)
pred_lgb = model_lgb.predict(X_val)
pred_proba = model_lgb.predict_proba(X_val)

# Plot feature importance
lgb.plot_importance(model_lgb, figsize=(10, 8))
plt.show()

In [ ]:
import optuna

def objective(trial):
    # Define the thresholds for each class
    thresholds = {}
    for i in range(num_classes):
        thresholds[f'threshold_{i}'] = trial.suggest_uniform(f'threshold_{i}', 0.0, 1.0)

    # Apply the thresholds to convert probabilities to predictions
    y_pred = apply_thresholds(pred_proba, thresholds)

    # Calculate accuracy
    accuracy = accuracy_score(y_val, y_pred)
    return accuracy  

def apply_thresholds(y_proba, thresholds):
    # Apply the specified thresholds to convert probabilities to predicted labels
    y_pred_labels = np.argmax(y_proba, axis=1)
    for i in range(y_proba.shape[1]):
        y_pred_labels[y_proba[:, i] > thresholds[f'threshold_{i}']] = i

    return y_pred_labels

num_classes = 7
pred_proba = pred_proba
y_val = y_val  

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=100)

# Get the best thresholds
best_thresholds = study.best_params
print("best_thresholds:", best_thresholds)

In [ ]:
parameters = {'threshold_0': 0.725, 
             'threshold_1': 0.616, 
             'threshold_2': 0.292, 
             'threshold_3': 0.315, 
             'threshold_4': 0.847, 
             'threshold_5': 0.689, 
             'threshold_6': 0.359}

In [ ]:
Best_Thresholds2= {'threshold_0': 0.724201213234911, 
                   'threshold_1': 0.6161299800571379, 
                   'threshold_2': 0.29138887902587174, 
                   'threshold_3': 0.3145837593497076, 
                   'threshold_4': 0.8469398340837189, 
                   'threshold_5': 0.6800824438387787, 
                   'threshold_6': 0.35886959729223455}

In [ ]:
test_label = model_lgb.predict_proba(X_test)
test_label = apply_thresholds(test_label, parameters)
pred = labelencoder.inverse_transform(test_label)
submission = pd.DataFrame({'id': test_data.id, 'NObeyesdad': pred})
#print(submission)
submission.to_csv('submission.csv', index=False)
submission['NObeyesdad'].value_counts()

In [ ]:
test_label = model_lgb.predict_proba(X_test)
test_label = apply_thresholds(test_label, Best_Thresholds2)
pred = labelencoder.inverse_transform(test_label)
submission = pd.DataFrame({'id': test_data.id, 'NObeyesdad': pred})
#print(submission)
submission.to_csv('submission1.csv', index=False)
submission['NObeyesdad'].value_counts()

NObeyesdad
Obesity_Type_III       2621
Obesity_Type_II        2139
Normal_Weight          2135
Obesity_Type_I         1990
Overweight_Level_II    1889
Insufficient_Weight    1700
Overweight_Level_I     1366

# Cross_validation:

from sklearn.model_selection import StratifiedKFold
from sklearn.base import clone

X = train_data.drop(['NObeyesdad'], axis=1)
y = train_data['NObeyesdad']
y = labelencoder.fit_transform(y)
X_test = test_data.drop(["id"],axis=1)
X_train, X_val, y_train, y_val = train_test_split(X,y,test_size=0.2,random_state=42)
# Initialize LightGBM model
#model = lgb.LGBMClassifier(**param,verbose=100)

# Initialize StratifiedKFold
skf = StratifiedKFold(n_splits=7, shuffle=True, random_state=42)

# Perform cross-validation
accuracies = []
for train_index, val_index in skf.split(X, y):
    X_train, X_val = X.iloc[train_index], X.iloc[val_index]
    y_train, y_val = y[train_index], y[val_index]
    
    # Clone the LightGBM model for each fold
    model = lgb.LGBMClassifier(**param)
    # Train the model
    model.fit(X_train, y_train,eval_set=[(X_val, y_val)])
    
    # Make predictions on the test set
    y_pred = model.predict(X_val)
    y_pred_pro = model.predict_proba(X_val)
    
    # Calculate accuracy
    accuracy = accuracy_score(y_val, y_pred)
    accuracies.append(accuracy)

# Calculate mean accuracy across folds
mean_accuracy = np.mean(accuracies)
print("Mean Accuracy:", mean_accuracy)

import optuna

def objective(trial):
    # Define the thresholds for each class
    thresholds = {}
    for i in range(num_classes):
        thresholds[f'threshold_{i}'] = trial.suggest_uniform(f'threshold_{i}', 0.0, 1.0)

    # Apply the thresholds to convert probabilities to predictions
    y_pred = apply_thresholds(y_pred_pro, thresholds)

    # Calculate accuracy
    accuracy = accuracy_score(y_val, y_pred)
    return accuracy  

def apply_thresholds(y_proba, thresholds):
    # Apply the specified thresholds to convert probabilities to predicted labels
    y_pred_labels = np.argmax(y_proba, axis=1)
    for i in range(y_proba.shape[1]):
        y_pred_labels[y_proba[:, i] > thresholds[f'threshold_{i}']] = i

    return y_pred_labels

num_classes = 7
y_pred_pro = y_pred_pro
y_val = y_val  

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=100)

# Get the best thresholds
best_thresholds = study.best_params
print("best_thresholds:", best_thresholds)

#X_test = test_data.drop(["id"],axis=1)
test_label = model.predict_proba(X_test)
test_label = apply_thresholds(test_label, best_thresholds)
pred = labelencoder.inverse_transform(test_label)
submission = pd.DataFrame({'id': test_data.id, 'NObeyesdad': pred})
#print(submission)
submission.to_csv('submission2.csv', index=False)
submission['NObeyesdad'].value_counts()